# 02 — Price Process Analysis

Inspect the simulated price dynamics:
- Mid price trajectories with factor correlation
- GARCH vol clustering
- MMPP-driven drift (micro-price mechanism)
- Bond similarity spillover in prices


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfq_sim.utils.io import load_all

data     = load_all('../data')
obs      = data['observable']
bond_meta = data['bond_metadata']

# Convert timestamp
obs['timestamp'] = pd.to_datetime(obs['timestamp'])
obs = obs.sort_values('timestamp')


In [ ]:
# Plot mid price trajectories for a sample of bonds
# I want to see: correlated sector moves, occasional jumps, spread variation

sample_bonds = bond_meta.groupby('sector').first().reset_index()['bond_id'].tolist()

fig, axes = plt.subplots(len(sample_bonds), 1, figsize=(14, 3*len(sample_bonds)),
                          sharex=True)

for ax, bond_id in zip(axes, sample_bonds):
    bond_data = obs[obs['bond_id'] == bond_id].copy()
    if len(bond_data) == 0:
        continue
    sector = bond_meta.loc[bond_meta['bond_id']==bond_id, 'sector'].values[0]
    ax.plot(bond_data['timestamp'], bond_data['mid_at_request'],
            linewidth=0.8, alpha=0.8)
    ax.set_ylabel(f'Bond {bond_id}\n({sector})', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
fig.suptitle('Mid Price Trajectories by Sector', y=1.01)
plt.tight_layout()
plt.savefig('../data/plots_price_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Check correlation structure between bond prices by sector
# Bonds in the same sector should be more correlated than cross-sector

# Pivot to get price matrix
price_pivot = obs.groupby(['timestamp', 'bond_id'])['mid_at_request'].mean().unstack()
price_returns = price_pivot.diff().dropna()

# Select bonds with enough data
valid_bonds = price_returns.columns[price_returns.notna().sum() > 50]
corr_matrix = price_returns[valid_bonds].corr()

# Get sector labels for ordering
sector_labels = bond_meta.set_index('bond_id').loc[
    [b for b in valid_bonds if b in bond_meta['bond_id'].values], 'sector'
]
sector_order = sector_labels.sort_values().index

fig, ax = plt.subplots(figsize=(10, 8))
bond_order_list = [b for b in sector_order if b in corr_matrix.columns]
if bond_order_list:
    sns.heatmap(
        corr_matrix.loc[bond_order_list, bond_order_list],
        ax=ax, cmap='RdBu_r', vmin=-1, vmax=1,
        xticklabels=False, yticklabels=False,
        center=0,
    )
    ax.set_title('Price Return Correlation Matrix\n(bonds sorted by sector)')
    plt.tight_layout()
    plt.show()
    print("Cross-sector correlations should be lower than within-sector.")


In [ ]:
# Spread dynamics -- should widen with vol and inventory
# Check: do high-inventory periods have wider spreads?
obs_wins = obs[obs['outcome'] == 'WIN'].copy()

if len(obs_wins) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Spread vs absolute inventory
    axes[0].scatter(
        obs['inventory_at_request'].abs(),
        obs['spread_at_request'],
        alpha=0.05, s=5, color='steelblue',
    )
    axes[0].set_xlabel('|Inventory| ($MM)')
    axes[0].set_ylabel('Bid-Ask Spread')
    axes[0].set_title('Spread vs Inventory Pressure')

    # Spread by liquidity tier
    obs.boxplot(column='spread_at_request', by='bond_liquidity_tier', ax=axes[1])
    axes[1].set_title('Spread Distribution by Liquidity Tier')
    axes[1].set_xlabel('Liquidity Tier')

    plt.tight_layout()
    plt.show()
    # Tier 3 (illiquid) should have wider spreads than Tier 1
